In [0]:
%run ./adls_auth

In [0]:
%run ./control_table

In [0]:
from datetime import datetime

SOURCE_NAME = "open_meteo_weather"

dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("pipeline_run_id", "")

start_date = dbutils.widgets.get("start_date").strip()
end_date = dbutils.widgets.get("end_date").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

started_at = datetime.utcnow()
partition_key = f"{start_date}_to_{end_date}"
file_path = f"abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/weather_raw/batch_{start_date}_to_{end_date}"

In [0]:
try:
    if not start_date or not end_date:
        raise ValueError("start_date/end_date not provided.")

    try:

        weather_df = spark.read.option("multiline", "true").json(file_path)
        

        result = weather_df.selectExpr("coalesce(size(hourly.time), 0) as n").collect()
        rows_written = result[0]["n"] if result else 0
        

        rows_written = max(rows_written, 0)
        status = "SUCCESS" if rows_written > 0 else "SUCCESS_BUT_EMPTY"
        
    except Exception as read_err:
        print(f"ERROR reading landed weather file at {file_path}: {read_err}")
        status = "SUCCESS_BUT_UNREADABLE"
        rows_written = 0

    log_ingestion_event(
    spark=spark,
    batch_id=pipeline_run_id,         
    source_name=SOURCE_NAME,
    partition_key=partition_key,
    status=status,
    rows_written=rows_written,
    started_at=started_at
        )

    if status == "SUCCESS":
        update_watermark(spark, SOURCE_NAME, end_date)
        print(f"Watermark advanced to {end_date}. Logged {rows_written} hourly readings for {partition_key}.")
    else:
        print(f"NOT advancing watermark — status={status} for {partition_key}. Will retry this same chunk next run.")

except Exception as e:
    print(f"Failed to finalize weather ingestion for {partition_key}: {e}")
    raise

In [0]:
# display(spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/watermark_log"))

In [0]:
# display(spark.read.format("delta").load("abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/ingestion_log"))